In [ ]:
import openeo

In [ ]:
connection = openeo.connect(url="openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

# 1. Definition of area and time of interest

In [ ]:
lakeName = "Flughafensee"
lakeBbox = {
    "west": 13.28098998347257, 
    "south": 52.56464939209237, 
    "east": 13.294593898905942, 
    "north": 52.57090962096577
}
startDate = "2020-04-01" # as string ("yyyy-mm-dd")
endDate = "2020-10-31" # as string ("yyyy-mm-dd")

In [ ]:
title = "NDTrI_" + lakeName + "_" + startDate + "_" + endDate

# 2. Setting up the job for OpenEO

The following section only defines the process archicecture that will be passed on to the OpenEO platform

## 2.1 Filter Collection
Select Sentinel-2 collection, spatial and temporal filtering

In [ ]:
s2cube = connection.load_collection("SENTINEL2_L2A")

In [ ]:
# Bounding box ()
s2cube = s2cube.filter_bbox(
    west = lakeBbox["west"], 
    south = lakeBbox["south"], 
    east = lakeBbox["east"], 
    north = lakeBbox["north"]
) 

In [ ]:
s2cube = s2cube.filter_temporal(
    start_date = startDate, 
    end_date = endDate
)
s2cube = s2cube.filter_bands(["B02", "B05", "SCL"])

In [ ]:
# single band data cubes
SCL = s2cube.band("SCL")
b2 = s2cube.band("B02")
b5 = s2cube.band("B05")

## 2.2 Quality control

Only use pixels which are mainly (> 80%) defined as water
That excludes pixels wich are often affected by surrounding conditions like casted shadows or misclassification due to sediment issues (shallow clean water).Furthermore, lakes that are often covered by duckweed are excluded.

In [ ]:
# masking clouds, cloud shadows, snow and tropogrophy shadwos
non_cloudy_mask = ((SCL == 2) | (SCL == 3) | (SCL == 7) | (SCL == 8) | (SCL == 9) | (SCL == 10) |  (SCL == 11)) 
non_cloudy_count = s2cube.filter_bands(bands = ["SCL"]).mask(non_cloudy_mask).reduce_dimension(reducer = "count", dimension = "t") * 1.0

# masking everything but water
water_mask =  ~ (SCL == 6)
water_count = s2cube.filter_bands(bands = ["SCL"]).mask(water_mask).reduce_dimension(reducer = "count", dimension = "t") * 1.0

# identify pixels that in more than 80% identified as water by SCL band
high_quality_pixels = (water_count / non_cloudy_count) > 0.8

## 2.3 Calculation of the Normalized Different Trophic Index (NDTrI)

The index is first calculated for each pixel, and is filterd for 
 - water pixels only
 - pixels that passed the quality control.
 
Subsequently monthly means are calculated wich are then further aggregated to a seasonal mean.

In [ ]:
ndtri_image = ((b5 - b2) / (b5 + b2)).mask(water_mask).mask(~ high_quality_pixels)

In [ ]:
ndtri_month = ndtri_image.aggregate_temporal_period(period = "month", reducer = "mean")

In [ ]:
ndtri = ndtri_month.reduce_dimension(reducer = "mean", dimension = "t") 

# 3. Execute the job on OpenEU and download Data (GeoTiff)

In [ ]:
job = ndtri.execute_batch(
    out_format = "GTiff", 
    title = title
)

In [ ]:
job.get_results().download_file("ndtri-results/" + title + ".tiff")

# 3b. Alternative execution to get netCDF file


In [ ]:
job = ndtri.execute_batch(
    out_format = "netcdf", 
    title = title
)

In [ ]:
job.get_results().download_file("ndtri-results/" + title + ".nc")